
## 1. Setup

Extract the source tables from the current Google Sheets export and persist
them as raw CSV files.

This script represents the **Bronze layer** of the medallion architecture.

Current source:

    Google Sheets -> Excel export -> Bronze CSV

Later, the Excel input can be replaced by the Google Sheets API without
changing the downstream Silver transformation script.

### 1.1 Paramaters

In [3]:
from pathlib import Path

import pandas as pd


# Pipeline paths
EXECUTION_DIR = (
    Path(__file__).resolve().parent
    if "__file__" in globals()
    else Path.cwd()
)
PIPELINES_DIR = next(
    candidate
    for directory in (EXECUTION_DIR, *EXECUTION_DIR.parents)
    for candidate in (directory, directory / "data-pipelines")
    if (candidate / "landing").is_dir()
)
INPUT_FILE = PIPELINES_DIR / "landing" / "expenses-tracker-potencial.xlsx"
BRONZE_DIR = PIPELINES_DIR / "database" / "bronze"

SOURCE_SHEETS = ["Trips", "Travelers", "Expenses", "Payments"]
print(INPUT_FILE)
print(BRONZE_DIR)

c:\GitHub\trip-expense-management\data-pipelines\landing\expenses-tracker-potencial.xlsx
c:\GitHub\trip-expense-management\data-pipelines\database\bronze


### 1.2 Source configuration

The workbook currently contains a title row before the actual table header,
so the second row (`header=1`) is used when reading each sheet.

The Bronze layer should preserve the source data as closely as possible.
Business transformations belong in the Silver layer.

In [4]:
def read_source_table(
    input_file: Path,
    sheet_name: str,
) -> pd.DataFrame:
    """Read one source table from the exported workbook."""
    if not input_file.exists():
        raise FileNotFoundError(f"Source file not found: {input_file}")

    return pd.read_excel(
        input_file,
        sheet_name=sheet_name,
        header=1,
    )

## 2. Data Sources

The source system currently consists of four Google Sheets tables:

- `Trips`
- `Travelers`
- `Expenses`
- `Payments`

No derived tables are created here.

### 2.1 Source Definition

In [5]:
source_tables = {}

for sheet_name in SOURCE_SHEETS:
    source_tables[sheet_name] = read_source_table(
        INPUT_FILE,
        sheet_name,
    )

    print(
        f"{sheet_name}: "
        f"{source_tables[sheet_name].shape[0]} rows x "
        f"{source_tables[sheet_name].shape[1]} columns"
    )

Trips: 1 rows x 5 columns
Travelers: 6 rows x 2 columns
Expenses: 57 rows x 13 columns
Payments: 3 rows x 9 columns


### 2.2 Source inspection

Inspect the raw tables before writing them.

This is useful during development because the Bronze layer is our record of
what arrived from the source system.

In [6]:
for sheet_name, df in source_tables.items():
    print(f"\n--- {sheet_name} ---")
    print("Columns:", list(df.columns))
    display(df.head())


--- Trips ---
Columns: ['ID', 'Trip', 'Destination', 'Start Date', 'End Date']


,ID,Trip,Destination,Start Date,End Date
0,T001,SPS Sep 2026,San Pedro Sula,2026-09-04,2026-09-06



--- Travelers ---
Columns: ['ID', 'Traveler']


,ID,Traveler
0,P000,Todos
1,P001,Ken
2,P002,Jimmy
3,P003,Megan
4,P004,Kevin



--- Expenses ---
Columns: ['Trip ID', 'Trip', 'Reason', 'Date', 'Type', 'Product', 'Debtor', 'Lander', 'Currency', 'Rate', 'Monto', 'Monto LPS', 'Monto USD']


,Trip ID,Trip,Reason,Date,Type,Product,Debtor,Lander,Currency,Rate,Monto,Monto LPS,Monto USD
0,T001,SPS Sep 2026,Airbnb,2026-09-04,Amenidades,Airbnb,Todos,Jimmy,$,27,260.19,7025.13,260.190000
1,T001,SPS Sep 2026,Wendys,2026-09-04,Restaurantes,BAC MUSH M,Ken,Jimmy,L,27,239.00,239.00,8.851852
2,T001,SPS Sep 2026,Wendys,2026-09-04,Restaurantes,DAVE C,Jimmy,Jimmy,L,27,219.00,219.00,8.111111
3,T001,SPS Sep 2026,Wendys,2026-09-04,Restaurantes,AGRANDADO,Ken,Jimmy,L,27,26.00,26.00,0.962963
4,T001,SPS Sep 2026,Wendys,2026-09-04,Restaurantes,#8 10 NUGG,Megan,Jimmy,L,27,189.00,189.00,7.000000



--- Payments ---
Columns: ['Trip ID', 'Trip', 'Debtor', 'Lander', 'Currency', 'Rate', 'Monto', 'Monto LPS', 'Monto USD']


,Trip ID,Trip,Debtor,Lander,Currency,Rate,Monto,Monto LPS,Monto USD
0,T001,SPS Sep 2026,Kevin,Jimmy,L,27,4414.09,4414.09,163.484815
1,T001,SPS Sep 2026,Megan,Jimmy,L,27,2098.00,2098.00,77.703704
2,T001,SPS Sep 2026,Ken,Jimmy,L,27,3000.00,3000.00,111.111111


## 3. Transformations

There are intentionally **no business transformations** in the Bronze
extraction.

We only remove completely empty rows/columns that can be introduced by the
spreadsheet export. The actual business logic is handled by the Silver
transformation script.

In [7]:
def remove_empty_structure(df: pd.DataFrame) -> pd.DataFrame:
    """Remove completely empty rows and columns from the export."""
    return df.dropna(axis=0, how="all").dropna(axis=1, how="all").copy()


bronze_tables = {
    name: remove_empty_structure(df)
    for name, df in source_tables.items()
}

## 4. Validations

Bronze validation is intentionally lightweight.

We validate that all expected source tables exist and contain data, but we
do not enforce business rules here. Bronze should reflect the source, even
when the source contains data that later needs to be rejected or corrected.

In [8]:
assert set(bronze_tables) == set(SOURCE_SHEETS)

for sheet_name, df in bronze_tables.items():
    assert not df.empty, f"{sheet_name} is empty."

print("Bronze source validation passed.")

Bronze source validation passed.


## 5. Data Writing

Write one CSV per source table.

Output:

    bronze/
    ├── trips.csv
    ├── travelers.csv
    ├── expenses.csv
    └── payments.csv

In [9]:
def write_bronze_tables(
    tables: dict[str, pd.DataFrame],
    output_dir: Path,
) -> None:
    """Persist source tables as Bronze CSV files."""
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, df in tables.items():
        output_file = output_dir / f"{name.lower()}.csv"
        df.to_csv(output_file, index=False)
        print(f"Wrote {output_file}")


write_bronze_tables(bronze_tables, BRONZE_DIR)

print("\nBronze extraction completed successfully.")

Wrote c:\GitHub\trip-expense-management\data-pipelines\database\bronze\trips.csv
Wrote c:\GitHub\trip-expense-management\data-pipelines\database\bronze\travelers.csv
Wrote c:\GitHub\trip-expense-management\data-pipelines\database\bronze\expenses.csv
Wrote c:\GitHub\trip-expense-management\data-pipelines\database\bronze\payments.csv

Bronze extraction completed successfully.
